In [1]:
## 1. Data Loading & Understanding
## 2. Data Cleaning
## 3. MRR Decomposition Engine
## 4. Cohort & Retention Analysis
## 5. Opportunity Identification
## 6. Insights & Recommendations
## 7. Export for Dashboard

In [2]:
## 1. Data Loading & Understanding

In [3]:
import pandas as pd

accounts = pd.read_csv("ravenstack_accounts.csv")
subscription = pd.read_csv("ravenstack_subscriptions.csv")
churn_events = pd.read_csv("ravenstack_churn_events.csv")
feature_usage = pd.read_csv("ravenstack_feature_usage.csv")
support_tickets = pd.read_csv("ravenstack_support_tickets.csv")

# OVERVIEW
print("==== ACCOUNTS ====")
print(accounts.shape)
print(accounts.columns.tolist())
print(accounts.head(5))
print('\n')

print("==== SUBSCRIPTION ====")
print(subscription.shape)
print(subscription.columns.tolist())
print(subscription.head(5))
print('\n')

print("=== CHURN EVENTS ===")
print(churn_events.shape)
print(churn_events.columns.tolist())
print(churn_events.head(5))
print("\n")

print("=== FEATURE USAGE ===")
print(feature_usage.shape)
print(feature_usage.columns.tolist())
print(feature_usage.head(5))
print("\n")

print("=== SUPPORT TICKETS ===")
print(support_tickets.shape)
print(support_tickets.columns.tolist())
print(support_tickets.head(3))

==== ACCOUNTS ====
(500, 10)
['account_id', 'account_name', 'industry', 'country', 'signup_date', 'referral_source', 'plan_tier', 'seats', 'is_trial', 'churn_flag']
  account_id account_name    industry country signup_date referral_source  \
0   A-2e4581    Company_0      EdTech      US  16-10-2024         partner   
1   A-43a9e3    Company_1     FinTech      IN  17-08-2023           other   
2   A-0a282f    Company_2    DevTools      US  27-08-2024         organic   
3   A-1f0ac7    Company_3  HealthTech      UK  27-08-2023           other   
4   A-ce550d    Company_4  HealthTech      US  27-10-2024           event   

    plan_tier  seats  is_trial  churn_flag  
0       Basic      9     False       False  
1       Basic     18     False        True  
2       Basic      1     False       False  
3       Basic     24      True       False  
4  Enterprise     35     False        True  


==== SUBSCRIPTION ====
(5000, 14)
['subscription_id', 'account_id', 'start_date', 'end_date', 'plan_

In [4]:
## 2. Data Cleaning

In [5]:
# for SUBSCRIPTION:
print("=== SUBSCRIPTION INFO ===")
print(subscription.info() ,'\n')

print("=== MISSING VALUES ===")
print(subscription.isnull().sum() ,'\n')

print("=== DATA TYPES ===")
print(subscription.dtypes ,'\n')

print("==== UNIQUE PLAN_TIER ===")
print(subscription['plan_tier'].value_counts())

print("=== UNIQUE BILLING FREQUENCY ===")
print(subscription['billing_frequency'].value_counts())

print("=== Trial vs Paid ===")
print(subscription['is_trial'].value_counts())

print("=== Active vs Churned (User end_date) ===")
print("Active (end_date null)", subscription['end_date'].isnull().sum())
print("Churned (end_date not_null)", subscription['end_date'].notnull().sum())

=== SUBSCRIPTION INFO ===
<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   subscription_id    5000 non-null   str  
 1   account_id         5000 non-null   str  
 2   start_date         5000 non-null   str  
 3   end_date           486 non-null    str  
 4   plan_tier          5000 non-null   str  
 5   seats              5000 non-null   int64
 6   mrr_amount         5000 non-null   int64
 7   arr_amount         5000 non-null   int64
 8   is_trial           5000 non-null   bool 
 9   upgrade_flag       5000 non-null   bool 
 10  downgrade_flag     5000 non-null   bool 
 11  churn_flag         5000 non-null   bool 
 12  billing_frequency  5000 non-null   str  
 13  auto_renew_flag    5000 non-null   bool 
dtypes: bool(5), int64(3), str(6)
memory usage: 569.7 KB
None 

=== MISSING VALUES ===
subscription_id         0
account_id              0
sta

In [6]:
# 1) Converting dates to datetime  
subscription['end_date'] = pd.to_datetime(subscription['end_date'], dayfirst=True, errors='coerce')
subscription['start_date'] = pd.to_datetime(subscription['start_date'], dayfirst=True, errors='coerce')

# 2) Making a clean status column
subscription['status'] = subscription['end_date'].apply(lambda x: 'Churned' if pd.notnull(x) else 'Active')

# 3) Creating is_paid column
subscription['is_paid']=~subscription['is_trial']

# Basic Checks after Cleaning
print('=== Date Conversion ===')
print(subscription[['start_date','end_date']].dtypes)
print("\nNull start_date:", subscription['start_date'].isnull().sum())
print("Null end_date:", subscription['end_date'].isnull().sum())

print('\n=== Status Distribution ===')
print(subscription['status'].value_counts())

print('\n=== Paid vs Trial ===')
print(subscription['is_paid'].value_counts())

print("\n=== cleaned data ===")
print(subscription[['subscription_id', 'account_id', 'start_date', 'end_date', 'plan_tier', 
                     'mrr_amount', 'status', 'is_paid']].head(8))

=== Date Conversion ===
start_date    datetime64[us]
end_date      datetime64[us]
dtype: object

Null start_date: 0
Null end_date: 4854

=== Status Distribution ===
status
Active     4854
Churned     146
Name: count, dtype: int64

=== Paid vs Trial ===
is_paid
True     4222
False     778
Name: count, dtype: int64

=== cleaned data ===
  subscription_id account_id start_date   end_date   plan_tier  mrr_amount  \
0        S-8cec59   A-3c1a3f 2023-12-23 2024-12-04  Enterprise        2786   
1        S-0f6f44   A-9b9fe9 2024-06-11        NaT         Pro         833   
2        S-51c0d1   A-659280 2024-11-25        NaT  Enterprise           0   
3        S-f81687   A-e7a1e2 2024-11-23        NaT  Enterprise         995   
4        S-cff5a2   A-ba6516 2024-01-10        NaT  Enterprise        5373   
5        S-4b9b13   A-fa2041 2024-08-13        NaT         Pro         735   
6        S-dceac6   A-417d2f 2023-12-30        NaT  Enterprise         796   
7        S-8cad7b   A-5f2961 2024-12-23

C:\Users\kashy\AppData\Local\Temp\ipykernel_17428\287382613.py:3: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  subscription['start_date'] = pd.to_datetime(subscription['start_date'], dayfirst=True, errors='coerce')


In [7]:
# Look at the original end_date values that are not null
print("=== Sample of non-null end_date (original style) ===")
print(subscription.loc[subscription['end_date'].notnull(), 'end_date'])

=== Sample of non-null end_date (original style) ===
0      2024-12-04
16     2024-03-08
64     2024-11-12
73     2024-04-12
86     2024-04-09
          ...    
4756   2024-12-10
4810   2024-05-12
4831   2024-11-12
4952   2024-04-08
4956   2024-11-12
Name: end_date, Length: 146, dtype: datetime64[us]


In [8]:
# Keeping only Paid Subscriptions for revenue analysis 
paid_subs = subscription[subscription['is_paid']].copy()

In [9]:
# Creating start and end month
paid_subs['start_month'] = paid_subs['start_date'].dt.to_period('M')

paid_subs['end_month'] = paid_subs['end_date'].dt.to_period('M')

In [10]:
print("Paid_subs shape:", paid_subs.shape)
print("\nSample:")
print(paid_subs[['subscription_id', 'account_id', 'start_date', 'end_date', 
                 'mrr_amount', 'plan_tier', 'status', 'start_month', 'end_month']].head(8))

Paid_subs shape: (4222, 18)

Sample:
  subscription_id account_id start_date   end_date  mrr_amount   plan_tier  \
0        S-8cec59   A-3c1a3f 2023-12-23 2024-12-04        2786  Enterprise   
1        S-0f6f44   A-9b9fe9 2024-06-11        NaT         833         Pro   
3        S-f81687   A-e7a1e2 2024-11-23        NaT         995  Enterprise   
4        S-cff5a2   A-ba6516 2024-01-10        NaT        5373  Enterprise   
5        S-4b9b13   A-fa2041 2024-08-13        NaT         735         Pro   
6        S-dceac6   A-417d2f 2023-12-30        NaT         796  Enterprise   
7        S-8cad7b   A-5f2961 2024-12-23        NaT         304       Basic   
8        S-24796e   A-cc8c8f 2024-11-27        NaT        4577  Enterprise   

    status start_month end_month  
0  Churned     2023-12   2024-12  
1   Active     2024-06       NaT  
3   Active     2024-11       NaT  
4   Active     2024-01       NaT  
5   Active     2024-08       NaT  
6   Active     2023-12       NaT  
7   Active     

In [11]:
# === Date Range of Data ===
print('Earliest Start: ', paid_subs['start_date'].min())
print('Latest: ', paid_subs['end_date'].max())

Earliest Start:  2023-01-09 00:00:00
Latest:  2024-12-12 00:00:00


In [12]:
paid_subs.columns

Index(['subscription_id', 'account_id', 'start_date', 'end_date', 'plan_tier',
       'seats', 'mrr_amount', 'arr_amount', 'is_trial', 'upgrade_flag',
       'downgrade_flag', 'churn_flag', 'billing_frequency', 'auto_renew_flag',
       'status', 'is_paid', 'start_month', 'end_month'],
      dtype='str')

In [13]:
## 3. MRR Movement Engine

In [14]:
# New_mrr by month
new_mrr = paid_subs.groupby('start_month')['mrr_amount'].sum().reset_index().rename(columns={'mrr_amount':'new_mrr'})

# Converting period into timestamp easier handling
new_mrr['start_month'] = new_mrr['start_month'].dt.to_timestamp()

In [15]:
print("=== New MRR by Month ===")
print(new_mrr.head(12))
print("\nTotal months:", len(new_mrr))
print("Total New MRR across all time:", new_mrr['new_mrr'].sum())

=== New MRR by Month ===
   start_month  new_mrr
0   2023-01-01     4684
1   2023-02-01    11079
2   2023-03-01    25885
3   2023-04-01    41788
4   2023-05-01    85919
5   2023-06-01    74987
6   2023-07-01   120422
7   2023-08-01   165621
8   2023-09-01   116222
9   2023-10-01   182518
10  2023-11-01   217219
11  2023-12-01   267264

Total months: 24
Total New MRR across all time: 11338747


In [16]:
# Churned mrr (only for subscriptions which contain end_date)
churned_subs = paid_subs[paid_subs['status']=='Churned'].copy()

churned_mrr = churned_subs.groupby('end_month')['mrr_amount'].sum().reset_index().rename(columns={'mrr_amount':'churned_mrr'})

# Converting period to timestamp
churned_mrr['end_month'] = churned_mrr['end_month'].dt.to_timestamp()

In [17]:
print('==== Churned MRR by Month ====')
print(churned_mrr.head(11))
print('\nTotal Months : ', len(churned_mrr))
print('Total Churned MRR : ', churned_mrr['churned_mrr'].sum())

==== Churned MRR by Month ====
    end_month  churned_mrr
0  2023-01-01         6368
1  2023-05-01          245
2  2023-06-01        15124
3  2023-09-01         1026
4  2024-01-01        16393
5  2024-02-01         7951
6  2024-03-01        77847
7  2024-04-01        13634
8  2024-05-01        11772
9  2024-06-01        25016
10 2024-07-01         9404

Total Months :  16
Total Churned MRR :  308523


In [18]:
# Count of Churned Customers by month
cnt_churned_by_month = churned_subs.groupby('end_month')['subscription_id'].count().reset_index().rename(columns={'subscription_id':'count_of_Churns'})
cnt_churned_by_month

,end_month,count_of_Churns
0,2023-01,1
1,2023-05,1
2,2023-06,2
3,2023-09,2
4,2024-01,9
5,2024-02,9
6,2024-03,19
7,2024-04,9
8,2024-05,5
9,2024-06,8


In [19]:
# Combining new_mrr and churned_mrr for better mrr_movement table

# Renaming to merge easily
new_mrr = new_mrr.rename(columns={'start_month':'month'})
churned_mrr = churned_mrr.rename(columns={'end_month':'month'})

# mrr_mmovement table creation
mrr_movement = pd.merge(new_mrr, churned_mrr , on='month', how='outer')

# filling null values
mrr_movement = mrr_movement.fillna(0)

# Sorting by month
mrr_movement = mrr_movement.sort_values('month').reset_index(drop=True)

print("=== MRR Movement (New + Churned) ===")
print(mrr_movement)

=== MRR Movement (New + Churned) ===
        month  new_mrr  churned_mrr
0  2023-01-01     4684       6368.0
1  2023-02-01    11079          0.0
2  2023-03-01    25885          0.0
3  2023-04-01    41788          0.0
4  2023-05-01    85919        245.0
5  2023-06-01    74987      15124.0
6  2023-07-01   120422          0.0
7  2023-08-01   165621          0.0
8  2023-09-01   116222       1026.0
9  2023-10-01   182518          0.0
10 2023-11-01   217219          0.0
11 2023-12-01   267264          0.0
12 2024-01-01   276933      16393.0
13 2024-02-01   365442       7951.0
14 2024-03-01   421916      77847.0
15 2024-04-01   445858      13634.0
16 2024-05-01   632549      11772.0
17 2024-06-01   537758      25016.0
18 2024-07-01   707816       9404.0
19 2024-08-01   648426       6024.0
20 2024-09-01   992366      40181.0
21 2024-10-01  1173608      33217.0
22 2024-11-01  1549040      10254.0
23 2024-12-01  2273427      34067.0


In [20]:
# Net New Mrr
mrr_movement['net_new_mrr'] = mrr_movement['new_mrr'] - mrr_movement['churned_mrr']

print('==== Mrr Movement with Net New Mrr ====')
print(mrr_movement)

==== Mrr Movement with Net New Mrr ====
        month  new_mrr  churned_mrr  net_new_mrr
0  2023-01-01     4684       6368.0      -1684.0
1  2023-02-01    11079          0.0      11079.0
2  2023-03-01    25885          0.0      25885.0
3  2023-04-01    41788          0.0      41788.0
4  2023-05-01    85919        245.0      85674.0
5  2023-06-01    74987      15124.0      59863.0
6  2023-07-01   120422          0.0     120422.0
7  2023-08-01   165621          0.0     165621.0
8  2023-09-01   116222       1026.0     115196.0
9  2023-10-01   182518          0.0     182518.0
10 2023-11-01   217219          0.0     217219.0
11 2023-12-01   267264          0.0     267264.0
12 2024-01-01   276933      16393.0     260540.0
13 2024-02-01   365442       7951.0     357491.0
14 2024-03-01   421916      77847.0     344069.0
15 2024-04-01   445858      13634.0     432224.0
16 2024-05-01   632549      11772.0     620777.0
17 2024-06-01   537758      25016.0     512742.0
18 2024-07-01   707816       

In [21]:
paid_subs.columns.to_list()

['subscription_id',
 'account_id',
 'start_date',
 'end_date',
 'plan_tier',
 'seats',
 'mrr_amount',
 'arr_amount',
 'is_trial',
 'upgrade_flag',
 'downgrade_flag',
 'churn_flag',
 'billing_frequency',
 'auto_renew_flag',
 'status',
 'is_paid',
 'start_month',
 'end_month']

In [22]:
# ==== Expansion MRR (upgrades) ====
expansion_mrr = paid_subs[paid_subs['upgrade_flag']==True].groupby('start_month')['mrr_amount'].sum().reset_index().rename(columns={'start_month':'month', 'mrr_amount':'expansion_mrr'})
expansion_mrr['month'] = expansion_mrr['month'].dt.to_timestamp()
print('==== Expansion MRR (upgrades) ====')
print(expansion_mrr)
print('Net Expansion Mrr : ', expansion_mrr['expansion_mrr'].sum())

# ==== Contraction MRR (downgrades) ====
contraction_mrr = paid_subs[paid_subs['downgrade_flag']==True].groupby('end_month')['mrr_amount'].sum().reset_index().rename(columns={'end_month':'month', 'mrr_amount':'contraction_mrr'})
contraction_mrr['month'] = contraction_mrr['month'].dt.to_timestamp()
print('\n==== Contraction MRR (downgrades) ====')
print(contraction_mrr)
print('Net Contracton Mrr : ', contraction_mrr['contraction_mrr'].sum())

==== Expansion MRR (upgrades) ====
        month  expansion_mrr
0  2023-01-01           3582
1  2023-02-01             76
2  2023-03-01           6004
3  2023-04-01           6733
4  2023-05-01          12139
5  2023-06-01          18402
6  2023-07-01          16381
7  2023-08-01          23614
8  2023-09-01          22924
9  2023-10-01          16480
10 2023-11-01          40658
11 2023-12-01          42839
12 2024-01-01          54563
13 2024-02-01          53086
14 2024-03-01          35685
15 2024-04-01          97594
16 2024-05-01          94577
17 2024-06-01          67514
18 2024-07-01          45770
19 2024-08-01          78996
20 2024-09-01         112238
21 2024-10-01         133339
22 2024-11-01         119623
23 2024-12-01         160180
Net Expansion Mrr :  1262997

==== Contraction MRR (downgrades) ====
       month  contraction_mrr
0 2024-04-01              345
1 2024-10-01              456
2 2024-12-01             1617
Net Contracton Mrr :  2418


In [23]:
# Merge everything into one table
mrr_full = mrr_movement.copy()

mrr_full = pd.merge(mrr_full, expansion_mrr, on='month', how='left')
mrr_full = pd.merge(mrr_full, contraction_mrr, on='month', how='left')

mrr_full = mrr_full.fillna(0)

# Reordering
mrr_full = mrr_full[['month', 'new_mrr', 'expansion_mrr', 'contraction_mrr', 'churned_mrr', 'net_new_mrr']]

print(mrr_full)

        month  new_mrr  expansion_mrr  contraction_mrr  churned_mrr  \
0  2023-01-01     4684           3582              0.0       6368.0   
1  2023-02-01    11079             76              0.0          0.0   
2  2023-03-01    25885           6004              0.0          0.0   
3  2023-04-01    41788           6733              0.0          0.0   
4  2023-05-01    85919          12139              0.0        245.0   
5  2023-06-01    74987          18402              0.0      15124.0   
6  2023-07-01   120422          16381              0.0          0.0   
7  2023-08-01   165621          23614              0.0          0.0   
8  2023-09-01   116222          22924              0.0       1026.0   
9  2023-10-01   182518          16480              0.0          0.0   
10 2023-11-01   217219          40658              0.0          0.0   
11 2023-12-01   267264          42839              0.0          0.0   
12 2024-01-01   276933          54563              0.0      16393.0   
13 202

In [24]:
# Calculating Net New Mrr
mrr_full['net_new_mrr'] = (mrr_full['new_mrr'] + mrr_full['expansion_mrr'] - mrr_full['churned_mrr'] - mrr_full['contraction_mrr'])

print('==== Final Mrr Movement Table ====')
print(mrr_full.round(0))

==== Final Mrr Movement Table ====
        month  new_mrr  expansion_mrr  contraction_mrr  churned_mrr  \
0  2023-01-01     4684           3582              0.0       6368.0   
1  2023-02-01    11079             76              0.0          0.0   
2  2023-03-01    25885           6004              0.0          0.0   
3  2023-04-01    41788           6733              0.0          0.0   
4  2023-05-01    85919          12139              0.0        245.0   
5  2023-06-01    74987          18402              0.0      15124.0   
6  2023-07-01   120422          16381              0.0          0.0   
7  2023-08-01   165621          23614              0.0          0.0   
8  2023-09-01   116222          22924              0.0       1026.0   
9  2023-10-01   182518          16480              0.0          0.0   
10 2023-11-01   217219          40658              0.0          0.0   
11 2023-12-01   267264          42839              0.0          0.0   
12 2024-01-01   276933          54563     

C:\Users\kashy\AppData\Local\Temp\ipykernel_17428\965124803.py:5: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  print(mrr_full.round(0))


In [25]:
## 4. Quick Ratio

In [26]:
# Quick Ratio
mrr_full['quick_ratio'] = (mrr_full['new_mrr'] + mrr_full['expansion_mrr'])/(mrr_full['churned_mrr'] + mrr_full['contraction_mrr']).replace(0,0.01)

mrr_full['quick_ratio'] = mrr_full['quick_ratio'].round(2)

print('==== Mrr Movement With Quick Ratio ====')
print(mrr_full)

==== Mrr Movement With Quick Ratio ====
        month  new_mrr  expansion_mrr  contraction_mrr  churned_mrr  \
0  2023-01-01     4684           3582              0.0       6368.0   
1  2023-02-01    11079             76              0.0          0.0   
2  2023-03-01    25885           6004              0.0          0.0   
3  2023-04-01    41788           6733              0.0          0.0   
4  2023-05-01    85919          12139              0.0        245.0   
5  2023-06-01    74987          18402              0.0      15124.0   
6  2023-07-01   120422          16381              0.0          0.0   
7  2023-08-01   165621          23614              0.0          0.0   
8  2023-09-01   116222          22924              0.0       1026.0   
9  2023-10-01   182518          16480              0.0          0.0   
10 2023-11-01   217219          40658              0.0          0.0   
11 2023-12-01   267264          42839              0.0          0.0   
12 2024-01-01   276933          54563

In [27]:
# Correcting Quick Ratio
denominator = mrr_full['churned_mrr'] + mrr_full['contraction_mrr']
mask = denominator > 0

mrr_full['quick_ratio'] = None

mrr_full.loc[mask,'quick_ratio'] =((mrr_full.loc[mask, 'new_mrr'] + mrr_full.loc[mask, 'expansion_mrr'])/denominator[mask]).round(2)

print('==== Cleaned Quick Ratio ====')
print(mrr_full)

==== Cleaned Quick Ratio ====
        month  new_mrr  expansion_mrr  contraction_mrr  churned_mrr  \
0  2023-01-01     4684           3582              0.0       6368.0   
1  2023-02-01    11079             76              0.0          0.0   
2  2023-03-01    25885           6004              0.0          0.0   
3  2023-04-01    41788           6733              0.0          0.0   
4  2023-05-01    85919          12139              0.0        245.0   
5  2023-06-01    74987          18402              0.0      15124.0   
6  2023-07-01   120422          16381              0.0          0.0   
7  2023-08-01   165621          23614              0.0          0.0   
8  2023-09-01   116222          22924              0.0       1026.0   
9  2023-10-01   182518          16480              0.0          0.0   
10 2023-11-01   217219          40658              0.0          0.0   
11 2023-12-01   267264          42839              0.0          0.0   
12 2024-01-01   276933          54563          

In [28]:
## 5. Churn Reason Analysis

In [29]:
# Churn Analysis

# Taking the highest MRR subscription per account
account_mrr = (paid_subs
               .sort_values('mrr_amount', ascending=False)
               .drop_duplicates(subset = 'account_id', keep = 'first')
               [['account_id', 'mrr_amount', 'plan_tier']]
              )

# Merging churn events with paid_subs 
churn_with_mrr = churn_events.merge(
    account_mrr [['account_id', 'mrr_amount', 'plan_tier']],
    on = 'account_id',
    how = 'left'
) 

# Final Churn Summary
churn_reasons_summary = (
    churn_with_mrr.groupby('reason_code')
    .agg(
        count = ('account_id', 'count'),
        total_refund = ('refund_amount_usd', 'sum'),
        total_churned_mrr = ('mrr_amount', 'sum')
    )
    .sort_values('count', ascending=False)
    .reset_index()
)

print('==== Churn Reasons with Revenue Impact ====')
print(churn_reasons_summary)

==== Churn Reasons with Revenue Impact ====
  reason_code  count  total_refund  total_churned_mrr
0    features    114       1905.90             819214
1      budget    104       1247.54             768677
2     support    104       1219.64             814893
3     unknown     95       1742.39             749997
4  competitor     92       1203.53             704613
5     pricing     91       1333.25             679112


In [30]:
## 6. Opportunity Analysis (Plan + Industry)

In [31]:
# Analysis By Plan Tier 

# Expansion by Plan Tier
expansion_by_plan = (
    paid_subs[paid_subs['upgrade_flag'] == True]
    .groupby('plan_tier')['mrr_amount']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={'mrr_amount': 'expansion_mrr'})
)

# Churn by Plan Tier
churn_by_plan = (
    paid_subs[paid_subs['status'] == 'Churned']
    .groupby('plan_tier')['mrr_amount']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={'mrr_amount': 'churned_mrr'})
)

print("=== Expansion MRR by Plan ===")
print(expansion_by_plan)
print("\n=== Churned MRR by Plan ===")
print(churn_by_plan)

=== Expansion MRR by Plan ===
    plan_tier  expansion_mrr
0  Enterprise         921768
1         Pro         281113
2       Basic          60116

=== Churned MRR by Plan ===
    plan_tier  churned_mrr
0  Enterprise       230641
1         Pro        58996
2       Basic        18886


In [38]:
plan_tier_summary = (pd.merge(
    expansion_by_plan,
    churn_by_plan,
    on = 'plan_tier',
    how = 'outer'
).fillna(0))

plan_tier_summary['net_expansion'] = plan_tier_summary['expansion_mrr'] - plan_tier_summary['churned_mrr']

plan_tier_summary = plan_tier_summary.sort_values('net_expansion', ascending=False).reset_index(drop=True)

print(plan_tier_summary)

    plan_tier  expansion_mrr  churned_mrr  net_expansion
0  Enterprise         921768       230641         691127
1         Pro         281113        58996         222117
2       Basic          60116        18886          41230


In [39]:
paid_subs.columns.to_list()

['subscription_id',
 'account_id',
 'start_date',
 'end_date',
 'plan_tier',
 'seats',
 'mrr_amount',
 'arr_amount',
 'is_trial',
 'upgrade_flag',
 'downgrade_flag',
 'churn_flag',
 'billing_frequency',
 'auto_renew_flag',
 'status',
 'is_paid',
 'start_month',
 'end_month']

In [40]:
# Analysis By Industry

# To get the Industry Column from Accounts
paid_with_industry = paid_subs.merge(
    accounts[['account_id', 'industry']],
    on = 'account_id',
    how = 'left'
)

# Expansion by Industry
expansion_by_indus = (paid_with_industry[paid_with_industry['upgrade_flag']==True]
    .groupby('industry')['mrr_amount']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns = {'mrr_amount':'expansion_mrr'})
                     )

# Churn by Industry
churn_by_indus= (
    paid_with_industry[paid_with_industry['downgrade_flag']==True]
        .groupby('industry')['mrr_amount']
        .sum()
        .sort_values(ascending=False)
        .reset_index()
        .rename(columns= {'mrr_amount':'churn_mrr'})
        )

print("=== Expansion MRR by Industry ===")
print(expansion_by_indus)
print("\n=== Churned MRR by Industry ===")
print(churn_by_indus)

=== Expansion MRR by Industry ===
        industry  expansion_mrr
0        FinTech         309726
1     HealthTech         283160
2  Cybersecurity         262191
3       DevTools         228139
4         EdTech         179781

=== Churned MRR by Industry ===
        industry  churn_mrr
0  Cybersecurity     122065
1       DevTools     112289
2        FinTech      93996
3     HealthTech      69540
4         EdTech      61476


In [41]:
# Industry Summary 
industry_summary = pd.merge(
    expansion_by_indus,
    churn_by_indus,
    on = 'industry',
    how = 'outer',
).fillna(0)

industry_summary['net_expansion'] = industry_summary['expansion_mrr'] - industry_summary['churn_mrr']

industry_summary = industry_summary.sort_values('net_expansion', ascending=False).reset_index(drop=True)

print("=== Industry Summary ===")
print(industry_summary)

=== Industry Summary ===
        industry  expansion_mrr  churn_mrr  net_expansion
0        FinTech         309726      93996         215730
1     HealthTech         283160      69540         213620
2  Cybersecurity         262191     122065         140126
3         EdTech         179781      61476         118305
4       DevTools         228139     112289         115850


In [ ]:
## 7. Key Insights
1. The company is growing well. Every month we are adding more revenue than we are losing.
2. Enterprise customers give us the most extra money through upgrades, but they also cost us the most when they leave.
3. FinTech and HealthTech customers are the best. They upgrade more and leave less.
4. Cybersecurity and DevTools customers upgrade a lot but also cancel a lot. This is risky.
5. Most customers are leaving because the product is missing some features they need. Support problems and budget issues are the next biggest reasons.
6. When fewer customers cancel, our growth becomes much stronger.

In [ ]:
## 8. Recommendations
1. First priority should be to improve the product features that customers are missing. This is the biggest reason people are cancelling.
2. We should give special attention to Enterprise customers. Losing even a few of them hurts revenue a lot.
3. Support quality needs to improve, especially for important customers.
4. We should focus more on getting and keeping FinTech and HealthTech customers since they are performing the best.
5. We need to talk to Cybersecurity and DevTools customers who cancelled to understand the real problems.

In [45]:
# Exporting Cleaned and optimized files
mrr_full.to_csv(r"D:\TOP_SECRETE\DATA ANALYTICS\Project_02\Cleaned Files\mrr_full.csv", index = False)

churn_reasons_summary.to_csv(r"D:\TOP_SECRETE\DATA ANALYTICS\Project_02\Cleaned Files\churn_reasons_summary.csv", index = False)

industry_summary.to_csv(r"D:\TOP_SECRETE\DATA ANALYTICS\Project_02\Cleaned Files\industry_summary.csv", index = False)

plan_tier_summary.to_csv(r"D:\TOP_SECRETE\DATA ANALYTICS\Project_02\Cleaned Files\plan_tier_summary.csv", index = False)
